# 面试问题：ELECTRA 的 Generator 与 Replaced Token Detection 怎样联合训练？

## 可直接复述的回答主线

1. ELECTRA 先让小 Generator 在 MASK 位置预测原词，再把预测 token 填回句子，Discriminator 判断每个普通位置是否真的被替换。
2. Generator 优化 selected-position MLM loss，Discriminator 优化全句普通 token 的 binary RTD loss，因此监督比只看 MASK 位置更密集。
3. RTD 标签必须由 corrupted_token != original_token 计算；被选中但 Generator 恰好采回原词时标签仍然是 original。
4. Generator 通常比 Discriminator 小，采样或 hard-negative corruption 要停止梯度，两个目标再按权重联合优化。
5. 本例不用现成 Transformer/ELECTRA，手写多头双向注意力、Encoder block、Generator、corruption 和 Discriminator。
6. 评测使用 balanced accuracy、replaced recall、generator MLM accuracy，并逐 token 展示概率而不是只展示 assert。
7. 生产还需动态 mask、随机采样温度、loss 权重、词表/特殊符号门禁、大语料去重、分布式训练和下游迁移验证。

后续实验会在同一批可读输入上依次展示基线、手写核心机制、训练过程、逐样本结果、失败修正与生产边界。

## 1. 真实案例与输入预览

案例包含 8 条中文电商搜索句，每条固定为 `[CLS] + 5 个普通词 + [SEP]`，并选择两个位置交给 Generator。训练时用 Generator 当前分数最高的“非原词”作受控 hard negative，使小批次始终包含 RTD 正例；评估会逐句打印原文、MASK 输入、替换结果和每个位置的 Discriminator 概率。

In [1]:
import math  # 汇总梯度、损失和分类指标。
import warnings  # 过滤本地 PyTorch 的无关兼容警告。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持输出聚焦预训练过程。
import torch  # 使用基础张量和自动微分手写 ELECTRA。
torch.manual_seed(451)  # 固定模型初始化和 hard-negative 轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程以提高复现性。
sentences = [["[CLS]", "用户", "搜索", "红色", "无线", "耳机", "[SEP]"], ["[CLS]", "客户", "查询", "北京", "门店", "库存", "[SEP]"], ["[CLS]", "推荐", "轻薄", "游戏", "笔记本", "电脑", "[SEP]"], ["[CLS]", "查找", "防水", "运动", "智能", "手表", "[SEP]"], ["[CLS]", "比较", "华为", "手机", "近期", "价格", "[SEP]"], ["[CLS]", "需要", "儿童", "学习", "平板", "电脑", "[SEP]"], ["[CLS]", "商家", "更新", "黑色", "运动", "鞋子", "[SEP]"], ["[CLS]", "买家", "收藏", "便携", "蓝牙", "音箱", "[SEP]"]]  # 定义八条真实可读电商句子。
selected_positions = [[2, 4], [1, 3], [2, 5], [1, 4], [3, 5], [2, 3], [1, 5], [3, 4]]  # 为每条句子选择两个非特殊 token 位置。
special_tokens = ["[PAD]", "[MASK]", "[CLS]", "[SEP]"]  # 定义 padding、MLM mask 和边界符号。
ordinary_tokens = sorted({token for sentence in sentences for token in sentence if token not in {"[CLS]", "[SEP]"}})  # 收集实际中文普通词。
vocabulary = special_tokens + ordinary_tokens  # 构造 Generator 和 Discriminator 共用编号空间。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立 token 到整数映射。
id_to_token = {index: token for token, index in token_to_id.items()}  # 建立整数到 token 映射。
pad_id = token_to_id["[PAD]"]  # 保存 padding 编号。
mask_id = token_to_id["[MASK]"]  # 保存 MLM mask 编号。
cls_id = token_to_id["[CLS]"]  # 保存句首编号。
sep_id = token_to_id["[SEP]"]  # 保存句尾编号。
input_ids = torch.tensor([[token_to_id[token] for token in sentence] for sentence in sentences], dtype=torch.long)  # 编码八条等长原始句子。
selected_mask = torch.zeros_like(input_ids, dtype=torch.bool)  # 创建 Generator 预测位置 mask。
for row_index, positions in enumerate(selected_positions):  # 逐句写入两个被选位置。
    selected_mask[row_index, positions] = True  # 标记当前句子的实际 MLM 位置。
masked_input_ids = input_ids.masked_fill(selected_mask, mask_id)  # 用 MASK 替换 Generator 可见输入中的选中词。
eligible_mask = input_ids.ne(cls_id) & input_ids.ne(sep_id) & input_ids.ne(pad_id)  # 只在五个普通 token 上计算 RTD 指标。
print(f"输入shape={tuple(input_ids.shape)} vocabulary={len(vocabulary)} selected={int(selected_mask.sum().item())} eligible={int(eligible_mask.sum().item())}")  # 展示真实批次 schema。
for index in range(len(sentences)):  # 逐条预览原文和 Generator 输入。
    original_text = " ".join(sentences[index])  # 连接当前原始 token。
    masked_text = " ".join(id_to_token[int(token)] for token in masked_input_ids[index])  # 解码当前 MASK 后输入。
    print(f"样本{index + 1} original={original_text}")  # 输出真实完整句子。
    print(f"       selected={selected_positions[index]} generator_input={masked_text}")  # 输出选中位置和 Generator 实际输入。

输入shape=(8, 7) vocabulary=42 selected=16 eligible=40
样本1 original=[CLS] 用户 搜索 红色 无线 耳机 [SEP]
       selected=[2, 4] generator_input=[CLS] 用户 [MASK] 红色 [MASK] 耳机 [SEP]
样本2 original=[CLS] 客户 查询 北京 门店 库存 [SEP]
       selected=[1, 3] generator_input=[CLS] [MASK] 查询 [MASK] 门店 库存 [SEP]
样本3 original=[CLS] 推荐 轻薄 游戏 笔记本 电脑 [SEP]
       selected=[2, 5] generator_input=[CLS] 推荐 [MASK] 游戏 笔记本 [MASK] [SEP]
样本4 original=[CLS] 查找 防水 运动 智能 手表 [SEP]
       selected=[1, 4] generator_input=[CLS] [MASK] 防水 运动 [MASK] 手表 [SEP]
样本5 original=[CLS] 比较 华为 手机 近期 价格 [SEP]
       selected=[3, 5] generator_input=[CLS] 比较 华为 [MASK] 近期 [MASK] [SEP]
样本6 original=[CLS] 需要 儿童 学习 平板 电脑 [SEP]
       selected=[2, 3] generator_input=[CLS] 需要 [MASK] [MASK] 平板 电脑 [SEP]
样本7 original=[CLS] 商家 更新 黑色 运动 鞋子 [SEP]
       selected=[1, 5] generator_input=[CLS] [MASK] 更新 黑色 运动 [MASK] [SEP]
样本8 original=[CLS] 买家 收藏 便携 蓝牙 音箱 [SEP]
       selected=[3, 4] generator_input=[CLS] 买家 收藏 [MASK] [MASK] 音箱 [SEP]


## 2. Baseline / 基线：所有 token 都判为 original

训练集每句会产生两个真实替换、三个未替换普通词。普通 accuracy 会受类别比例影响，所以这里使用 positive recall 与 negative recall 的平均值 balanced accuracy；全判 original 的 baseline 固定为 0.5，并且 replaced recall 为 0。

In [2]:
controlled_positive_count = int(selected_mask.sum().item())  # 受控 hard-negative 会让每个 selected 位置真正变化。
controlled_negative_count = int(eligible_mask.sum().item()) - controlled_positive_count  # 其余普通 token 保持原样。
baseline_replaced_recall = 0.0  # 全判 original 无法命中任何替换词。
baseline_original_recall = 1.0  # 全判 original 会命中所有未替换词。
baseline_balanced_accuracy = (baseline_replaced_recall + baseline_original_recall) / 2.0  # 计算类别均衡准确率。
baseline_plain_accuracy = controlled_negative_count / int(eligible_mask.sum().item())  # 同时展示容易误导的普通准确率。
print(f"Baseline all-original: positives={controlled_positive_count} negatives={controlled_negative_count}")  # 展示评估类别组成。
print(f"plain_accuracy={baseline_plain_accuracy:.3f} balanced_accuracy={baseline_balanced_accuracy:.3f} replaced_recall={baseline_replaced_recall:.3f}")  # 输出后续模型的同指标参照。

Baseline all-original: positives=16 negatives=24
plain_accuracy=0.600 balanced_accuracy=0.500 replaced_recall=0.000


## 3. 底层实现：双向多头注意力、Generator 与 Discriminator

下面不调用 `TransformerEncoder` 或 ELECTRA 库。注意力分数、head 拆分、residual、LayerNorm、FFN 都显式实现；Generator 较小并输出词表 logits，Discriminator 较大并为每个 token 输出一个 RTD logit。

In [3]:
class BidirectionalSelfAttention(torch.nn.Module):  # 手写不带 causal mask 的多头 self-attention。
    def __init__(self, dimension, heads):  # 初始化 QKV 和输出投影。
        super().__init__()  # 注册 PyTorch 参数。
        self.dimension = dimension  # 保存 hidden 总维度。
        self.heads = heads  # 保存注意力头数。
        self.head_dimension = dimension // heads  # 计算每头维度。
        self.query = torch.nn.Linear(dimension, dimension)  # 创建 query 投影。
        self.key = torch.nn.Linear(dimension, dimension)  # 创建 key 投影。
        self.value = torch.nn.Linear(dimension, dimension)  # 创建 value 投影。
        self.output = torch.nn.Linear(dimension, dimension)  # 创建多头拼接后的输出投影。
    def split_heads(self, values):  # 把 hidden 总维度拆成多个头。
        batch, length, _ = values.shape  # 读取批量与序列长度。
        return values.view(batch, length, self.heads, self.head_dimension).transpose(1, 2)  # 返回批乘头乘长度乘头维度。
    def forward(self, hidden, valid_mask):  # 对完整双向句子计算注意力。
        queries = self.split_heads(self.query(hidden))  # 投影并拆分 query。
        keys = self.split_heads(self.key(hidden))  # 投影并拆分 key。
        values = self.split_heads(self.value(hidden))  # 投影并拆分 value。
        scores = queries @ keys.transpose(-2, -1) / math.sqrt(self.head_dimension)  # 计算缩放点积分数。
        scores = scores.masked_fill(~valid_mask[:, None, None, :], -1.0e4)  # 屏蔽 padding key 但允许左右双向读取。
        weights = torch.softmax(scores, dim=-1)  # 把每个 query 的分数归一化。
        context = weights @ values  # 按概率聚合所有有效 token 的 value。
        merged = context.transpose(1, 2).contiguous().view(hidden.shape[0], hidden.shape[1], self.dimension)  # 拼回多头表示。
        return self.output(merged), weights  # 返回上下文 hidden 和可解释 attention。
class EncoderBlock(torch.nn.Module):  # 组合双向注意力与逐 token 前馈层。
    def __init__(self, dimension, heads):  # 初始化一个 pre-norm Encoder block。
        super().__init__()  # 注册所有子层。
        self.attention_norm = torch.nn.LayerNorm(dimension)  # 创建 attention 前归一化。
        self.attention = BidirectionalSelfAttention(dimension, heads)  # 创建手写双向多头注意力。
        self.feed_norm = torch.nn.LayerNorm(dimension)  # 创建 FFN 前归一化。
        self.feed_input = torch.nn.Linear(dimension, dimension * 2)  # 创建 FFN 升维层。
        self.feed_output = torch.nn.Linear(dimension * 2, dimension)  # 创建 FFN 降维层。
    def forward(self, hidden, valid_mask):  # 编码完整 token 序列。
        attention_output, attention_weights = self.attention(self.attention_norm(hidden), valid_mask)  # 双向聚合上下文。
        hidden = hidden + attention_output  # 应用 attention residual connection。
        feed_output = self.feed_output(torch.nn.functional.gelu(self.feed_input(self.feed_norm(hidden))))  # 执行非线性逐 token 变换。
        return hidden + feed_output, attention_weights  # 应用 FFN residual 并返回权重。
class TinyEncoder(torch.nn.Module):  # 组装 token/position embedding 和两层手写 Encoder。
    def __init__(self, vocabulary_size, dimension, heads, maximum_length=7):  # 初始化可配置大小的 Encoder。
        super().__init__()  # 注册完整 Encoder 参数。
        self.token_embedding = torch.nn.Embedding(vocabulary_size, dimension)  # 创建 token embedding。
        self.position_embedding = torch.nn.Embedding(maximum_length, dimension)  # 创建绝对位置 embedding。
        self.blocks = torch.nn.ModuleList([EncoderBlock(dimension, heads), EncoderBlock(dimension, heads)])  # 堆叠两个手写 Encoder block。
        self.final_norm = torch.nn.LayerNorm(dimension)  # 创建最终 hidden 归一化。
    def forward(self, token_ids, valid_mask):  # 把整数句子编码成上下文 hidden。
        positions = torch.arange(token_ids.shape[1], device=token_ids.device)[None, :]  # 构造每个 token 的位置编号。
        hidden = self.token_embedding(token_ids) + self.position_embedding(positions)  # 合并词义与位置信息。
        attention_trace = []  # 保存每层 attention 供解释。
        for block in self.blocks:  # 顺序运行两层 Encoder。
            hidden, weights = block(hidden, valid_mask)  # 执行当前双向注意力和 FFN。
            attention_trace.append(weights)  # 保存当前层注意力矩阵。
        return self.final_norm(hidden), attention_trace  # 返回最终上下文表示与两层权重。
class ElectraGenerator(torch.nn.Module):  # 定义较小的 masked-token Generator。
    def __init__(self, vocabulary_size):  # 初始化二十四维 Generator。
        super().__init__()  # 注册 PyTorch 模块。
        self.encoder = TinyEncoder(vocabulary_size, dimension=24, heads=4)  # 创建小容量双向 Encoder。
        self.output = torch.nn.Linear(24, vocabulary_size)  # 为每个位置输出完整词表 logits。
    def forward(self, token_ids, valid_mask):  # 预测 MASK 位置原词分布。
        hidden, attention = self.encoder(token_ids, valid_mask)  # 编码带 MASK 的句子。
        return self.output(hidden), {"hidden": hidden, "attention": attention}  # 返回 MLM logits 和中间量。
class ElectraDiscriminator(torch.nn.Module):  # 定义较大的逐 token RTD 分类器。
    def __init__(self, vocabulary_size):  # 初始化三十二维 Discriminator。
        super().__init__()  # 注册 PyTorch 模块。
        self.encoder = TinyEncoder(vocabulary_size, dimension=32, heads=4)  # 创建更大双向 Encoder。
        self.output = torch.nn.Linear(32, 1)  # 为每个位置输出 replaced logit。
    def forward(self, token_ids, valid_mask):  # 判断 corrupted sentence 中每个 token 是否被替换。
        hidden, attention = self.encoder(token_ids, valid_mask)  # 编码已经填回 token 的完整句子。
        return self.output(hidden).squeeze(-1), {"hidden": hidden, "attention": attention}  # 返回逐 token RTD logits 和中间量。
class TinyElectra(torch.nn.Module):  # 组合独立 Generator 和 Discriminator。
    def __init__(self, vocabulary_size):  # 初始化两套不同容量的 Encoder。
        super().__init__()  # 注册联合预训练模型。
        self.generator = ElectraGenerator(vocabulary_size)  # 创建小 Generator。
        self.discriminator = ElectraDiscriminator(vocabulary_size)  # 创建主 Discriminator。
def generator_informed_corruption(generator_logits, originals, positions, forbid_original=True):  # 用 Generator 分数构造填回句子。
    candidate_logits = generator_logits.detach().clone()  # 停止 corruption 路径梯度并复制可修改 logits。
    candidate_logits[:, :, :len(special_tokens)] = -1.0e4  # 禁止采到 PAD、MASK、CLS 或 SEP 等特殊 token。
    if forbid_original:  # 教学小批次需要稳定保留 RTD 正例。
        candidate_logits.scatter_(2, originals.unsqueeze(-1), -1.0e4)  # 排除原词并选择 Generator 当前最难的错误候选。
    replacements = candidate_logits.argmax(dim=-1)  # 选择当前最高分普通 token。
    return torch.where(positions, replacements, originals)  # 只在 selected 位置写回候选。
model = TinyElectra(len(vocabulary))  # 创建待联合训练的简化 ELECTRA。
generator_parameters = sum(parameter.numel() for parameter in model.generator.parameters())  # 统计小 Generator 参数量。
discriminator_parameters = sum(parameter.numel() for parameter in model.discriminator.parameters())  # 统计主 Discriminator 参数量。
print(f"Generator parameters={generator_parameters} Discriminator parameters={discriminator_parameters}")  # 展示 Generator 明确小于 Discriminator。

Generator parameters=12018 Discriminator parameters=18753


## 4. 真实联合训练：MLM loss + RTD loss

每一步先算 Generator MLM loss，再根据当前 Generator logits 选择最高分非原词 hard negative；随后用 `corrupted != original` 生成真实标签，计算所有普通 token 的 BCE。hard-negative 的离散选择停止梯度，但两个网络都由总 loss 在同一步更新。

In [4]:
valid_mask = input_ids.ne(pad_id)  # 标记 Encoder 中所有真实 token。
optimizer = torch.optim.Adam(model.parameters(), lr=0.006)  # 创建同时更新 Generator 和 Discriminator 的优化器。
training_history = []  # 保存联合训练 loss、MLM 和 RTD 指标。
latest_corrupted = None  # 预留最近一次实际 corruption。
latest_rtd_labels = None  # 预留最近一次相等性标签。
for step in range(801):  # 在八条真实句子上执行联合预训练。
    model.train()  # 开启训练模式。
    generator_logits, generator_debug = model.generator(masked_input_ids, valid_mask)  # 对 MASK 输入预测原词词表分布。
    mlm_loss = torch.nn.functional.cross_entropy(generator_logits[selected_mask], input_ids[selected_mask])  # 只在十六个 selected 位置计算 Generator loss。
    corrupted_ids = generator_informed_corruption(generator_logits, input_ids, selected_mask, forbid_original=True)  # 选择 Generator 当前最难的非原词候选并填回。
    rtd_labels = corrupted_ids.ne(input_ids).to(torch.float32)  # 根据实际 token 是否变化构造正确 RTD 标签。
    discriminator_logits, discriminator_debug = model.discriminator(corrupted_ids, valid_mask)  # 对完整 corrupted sentence 输出逐 token 分数。
    positive_count = rtd_labels[eligible_mask].sum()  # 统计当前真正被替换的正例数量。
    negative_count = eligible_mask.sum() - positive_count  # 统计当前保持原词的负例数量。
    positive_weight = negative_count / positive_count.clamp_min(1.0)  # 计算类别均衡 BCE 正例权重。
    rtd_loss = torch.nn.functional.binary_cross_entropy_with_logits(discriminator_logits[eligible_mask], rtd_labels[eligible_mask], pos_weight=positive_weight)  # 在全部普通 token 上计算密集 RTD loss。
    total_loss = mlm_loss + rtd_loss  # 以相同量级联合两个真实训练目标。
    optimizer.zero_grad(set_to_none=True)  # 清除上一轮两套网络梯度。
    total_loss.backward()  # 对 Generator MLM 和 Discriminator RTD 同时反向传播。
    generator_gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.generator.parameters() if parameter.grad is not None))  # 汇总 Generator 梯度二范数。
    discriminator_gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.discriminator.parameters() if parameter.grad is not None))  # 汇总 Discriminator 梯度二范数。
    optimizer.step()  # 应用一次联合 Adam 更新。
    latest_corrupted = corrupted_ids  # 保存当前真实 hard-negative 句子。
    latest_rtd_labels = rtd_labels  # 保存当前基于相等性的标签。
    if step % 200 == 0 or step == 800:  # 定期保存可读训练轨迹。
        with torch.no_grad():  # 统计指标时关闭梯度。
            mlm_accuracy = generator_logits[selected_mask].argmax(dim=-1).eq(input_ids[selected_mask]).to(torch.float32).mean().item()  # 计算 selected-position Generator top1。
            rtd_probabilities = torch.sigmoid(discriminator_logits)  # 把 RTD logits 转为 replaced 概率。
            rtd_predictions = rtd_probabilities.ge(0.5)  # 按固定阈值产生二分类结果。
            positive_recall = rtd_predictions[eligible_mask & rtd_labels.bool()].to(torch.float32).mean().item()  # 计算真正替换 token 召回率。
            negative_recall = (~rtd_predictions[eligible_mask & ~rtd_labels.bool()]).to(torch.float32).mean().item()  # 计算保持原词 token 召回率。
            balanced_accuracy = (positive_recall + negative_recall) / 2.0  # 平均两类召回避免比例偏差。
        training_history.append({"step": step, "total_loss": float(total_loss.item()), "mlm_loss": float(mlm_loss.item()), "rtd_loss": float(rtd_loss.item()), "mlm_accuracy": mlm_accuracy, "balanced_accuracy": balanced_accuracy, "generator_gradient_norm": generator_gradient_norm, "discriminator_gradient_norm": discriminator_gradient_norm})  # 保存完整联合训练证据。
print("ELECTRA训练轨迹=", training_history)  # 输出两个目标、两种准确率和两套梯度演化。
print("最近一次corrupted shape=", tuple(latest_corrupted.shape), "replaced count=", int(latest_rtd_labels[eligible_mask].sum().item()))  # 展示真正送入 Discriminator 的批次。

ELECTRA训练轨迹= [{'step': 0, 'total_loss': 4.732165336608887, 'mlm_loss': 3.68849778175354, 'rtd_loss': 1.0436677932739258, 'mlm_accuracy': 0.0625, 'balanced_accuracy': 0.3333333432674408, 'generator_gradient_norm': 1.2286567820283074, 'discriminator_gradient_norm': 1.1783516594266}, {'step': 200, 'total_loss': 0.003370467806234956, 'mlm_loss': 0.003017290960997343, 'rtd_loss': 0.0003531769325491041, 'mlm_accuracy': 1.0, 'balanced_accuracy': 1.0, 'generator_gradient_norm': 0.006404029025520331, 'discriminator_gradient_norm': 0.002439455254817437}, {'step': 400, 'total_loss': 0.0011188867501914501, 'mlm_loss': 0.0010077650658786297, 'rtd_loss': 0.00011112164793303236, 'mlm_accuracy': 1.0, 'balanced_accuracy': 1.0, 'generator_gradient_norm': 0.0022745586372517595, 'discriminator_gradient_norm': 0.0008087058219826275}, {'step': 600, 'total_loss': 0.0005638687289319932, 'mlm_loss': 0.000509605451952666, 'rtd_loss': 5.426325515145436e-05, 'mlm_accuracy': 1.0, 'balanced_accuracy': 1.0, 'generat

## 5. 最终逐 token 结果、结果解读与中间张量

最终重新生成 hard negative 并逐句输出。`R` 表示实际替换、`O` 表示保持原词；概率来自 Discriminator，而不是由 selected mask 冒充标签。

In [5]:
model.eval()  # 切换到确定性评估模式。
with torch.no_grad():  # 计算最终 Generator 与 Discriminator 输出。
    final_generator_logits, final_generator_debug = model.generator(masked_input_ids, valid_mask)  # 获得更新后的 MLM logits 和 attention。
    final_corrupted_ids = generator_informed_corruption(final_generator_logits, input_ids, selected_mask, forbid_original=True)  # 用最终 Generator 产生受控 hard negative。
    final_rtd_labels = final_corrupted_ids.ne(input_ids)  # 从实际 token equality 重新构造评估标签。
    final_discriminator_logits, final_discriminator_debug = model.discriminator(final_corrupted_ids, valid_mask)  # 对最终 corrupted batch 执行 RTD。
    final_rtd_probabilities = torch.sigmoid(final_discriminator_logits)  # 把 logits 转成 replaced 概率。
    final_rtd_predictions = final_rtd_probabilities.ge(0.5)  # 用零点五阈值分类。
generator_mlm_accuracy = final_generator_logits[selected_mask].argmax(dim=-1).eq(input_ids[selected_mask]).to(torch.float32).mean().item()  # 计算最终 Generator MLM top1。
replaced_recall = final_rtd_predictions[eligible_mask & final_rtd_labels].to(torch.float32).mean().item()  # 计算最终被替换词召回率。
original_recall = (~final_rtd_predictions[eligible_mask & ~final_rtd_labels]).to(torch.float32).mean().item()  # 计算最终原词召回率。
electra_balanced_accuracy = (replaced_recall + original_recall) / 2.0  # 汇总最终 balanced accuracy。
print(f"结果解读：baseline balanced_acc={baseline_balanced_accuracy:.3f} replaced_recall={baseline_replaced_recall:.3f}；ELECTRA balanced_acc={electra_balanced_accuracy:.3f} replaced_recall={replaced_recall:.3f}；Generator MLM acc={generator_mlm_accuracy:.3f}")  # 对比同数据指标。
for row_index in range(len(sentences)):  # 逐句展示 corruption 和 RTD 输出。
    original_tokens = [id_to_token[int(token)] for token in input_ids[row_index]]  # 解码原始句子。
    corrupted_tokens = [id_to_token[int(token)] for token in final_corrupted_ids[row_index]]  # 解码 Generator 填回后的句子。
    tags = ["R" if bool(value) else "O" for value in final_rtd_labels[row_index]]  # 把真实 equality 标签转成 R/O。
    probabilities = [round(float(value), 3) for value in final_rtd_probabilities[row_index]]  # 格式化逐 token replaced 概率。
    predictions = ["R" if bool(value) else "O" for value in final_rtd_predictions[row_index]]  # 格式化逐 token 二分类输出。
    print(f"样本{row_index + 1} original ={' '.join(original_tokens)}")  # 输出真实原文。
    print(f"       corrupted={' '.join(corrupted_tokens)}")  # 输出实际替换后的输入。
    print(f"       true_tag ={tags}")  # 输出基于相等性的真实标签。
    print(f"       rtd_prob ={probabilities} pred={predictions}")  # 输出模型概率和阈值结果。
first_selected_position = selected_positions[0][0]  # 选择样本一的第一个 MLM 位置作细查。
first_generator_probabilities = torch.softmax(final_generator_logits[0, first_selected_position], dim=-1)  # 计算该 MASK 位置的完整词表概率。
generator_top_probabilities, generator_top_indices = first_generator_probabilities.topk(5)  # 读取 Generator 最可能的五个词。
generator_top_candidates = [(id_to_token[int(index)], float(probability)) for probability, index in zip(generator_top_probabilities, generator_top_indices)]  # 转成人类可读候选。
print("样本1首个MASK的Generator top5=", generator_top_candidates)  # 展示 Generator 真实分布而非只显示 argmax。
print("样本1 Discriminator末层head0 attention=", torch.round(final_discriminator_debug["attention"][-1][0, 0] * 1000) / 1000)  # 展示双向上下文判断的中间权重。

结果解读：baseline balanced_acc=0.500 replaced_recall=0.000；ELECTRA balanced_acc=1.000 replaced_recall=1.000；Generator MLM acc=1.000
样本1 original =[CLS] 用户 搜索 红色 无线 耳机 [SEP]
       corrupted=[CLS] 用户 查找 红色 蓝牙 耳机 [SEP]
       true_tag =['O', 'O', 'R', 'O', 'R', 'O', 'O']
       rtd_prob =[0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0] pred=['O', 'O', 'R', 'O', 'R', 'O', 'R']
样本2 original =[CLS] 客户 查询 北京 门店 库存 [SEP]
       corrupted=[CLS] 商家 查询 轻薄 门店 库存 [SEP]
       true_tag =['O', 'R', 'O', 'R', 'O', 'O', 'O']
       rtd_prob =[0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0] pred=['O', 'R', 'O', 'R', 'O', 'O', 'R']
样本3 original =[CLS] 推荐 轻薄 游戏 笔记本 电脑 [SEP]
       corrupted=[CLS] 推荐 儿童 游戏 笔记本 价格 [SEP]
       true_tag =['O', 'O', 'R', 'O', 'O', 'R', 'O']
       rtd_prob =[0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0] pred=['O', 'O', 'R', 'O', 'O', 'R', 'R']
样本4 original =[CLS] 查找 防水 运动 智能 手表 [SEP]
       corrupted=[CLS] 搜索 防水 运动 无线 手表 [SEP]
       true_tag =['O', 'R', 'O', 'O', 'R', 'O', 'O']
       rtd_prob =[0.0, 1.0, 0.0, 0.0

## 6. 失败案例与修正：selected 不等于 replaced

标准 ELECTRA 从 Generator 分布采样时可能恰好采回原词。若直接把 `selected_mask` 当 RTD 标签，就会把没有变化的 token 错标成 replaced；正确标签永远来自填回后 token 与原 token 的逐位置比较。

In [6]:
edge_original = input_ids[0].clone()  # 复制一条原始句子用于边界案例。
edge_corrupted = edge_original.clone()  # 模拟 Generator 恰好采回原词的填回结果。
edge_selected = torch.zeros_like(edge_original, dtype=torch.bool)  # 创建当前样本的 selected 标志。
edge_selected[first_selected_position] = True  # 声明这个位置确实被送给 Generator 预测过。
wrong_edge_label = edge_selected.to(torch.long)  # 复现错误实现：把 selected 直接当 replaced 标签。
correct_edge_label = edge_corrupted.ne(edge_original).to(torch.long)  # 正确实现：比较实际填回 token 与原 token。
print(f"边界token={id_to_token[int(edge_original[first_selected_position])]} selected={bool(edge_selected[first_selected_position])} sampled_same_original={bool(edge_corrupted[first_selected_position] == edge_original[first_selected_position])}")  # 展示选中但没有变化的真实场景。
print(f"错误行为：label=selected -> {int(wrong_edge_label[first_selected_position])}，会制造假正例。")  # 输出错误标签一。
print(f"修正行为：label=(corrupted!=original) -> {int(correct_edge_label[first_selected_position])}，保持 original 标签。")  # 输出正确标签零。

边界token=搜索 selected=True sampled_same_original=True
错误行为：label=selected -> 1，会制造假正例。
修正行为：label=(corrupted!=original) -> 0，保持 original 标签。


## 7. 生产边界

受控 hard negative 让 8 条教学样本稳定可看，不是对大规模 ELECTRA 预训练的替代。生产要从 Generator 分布按温度随机采样并允许采回原词，按 mask ratio 与替换率校准 RTD loss，使用更小 Generator/更大 Discriminator 的算力配比，处理 whole-word masking、特殊 token、动态 padding、数据去重污染、混合精度与多机 checkpoint，并在 held-out RTD、下游任务和真实流量上监控收益。

In [7]:
electra_diagnostics = {"sentences": len(sentences), "selected_tokens": int(selected_mask.sum().item()), "eligible_tokens": int(eligible_mask.sum().item()), "generator_parameters": generator_parameters, "discriminator_parameters": discriminator_parameters, "baseline_balanced_accuracy": baseline_balanced_accuracy, "electra_balanced_accuracy": electra_balanced_accuracy, "replaced_recall": replaced_recall, "original_recall": original_recall, "generator_mlm_accuracy": generator_mlm_accuracy, "initial_total_loss": training_history[0]["total_loss"], "final_total_loss": training_history[-1]["total_loss"]}  # 汇总样本、模型规模、两个目标和类别均衡指标。
print("生产监控快照：", electra_diagnostics)  # 输出 ELECTRA 预训练应持续观察的核心信号。

生产监控快照： {'sentences': 8, 'selected_tokens': 16, 'eligible_tokens': 40, 'generator_parameters': 12018, 'discriminator_parameters': 18753, 'baseline_balanced_accuracy': 0.5, 'electra_balanced_accuracy': 1.0, 'replaced_recall': 1.0, 'original_recall': 1.0, 'generator_mlm_accuracy': 1.0, 'initial_total_loss': 4.732165336608887, 'final_total_loss': 0.00034068876993842423}


## 8. 最小回归测试

最后一格只保护真实样本规模、两套网络梯度、同指标收益、MLM 能力、模型容量关系和标签边界；所有教学输出都已在前面呈现。

In [8]:
assert len(sentences) >= 5 and input_ids.shape == (8, 7) and int(selected_mask.sum().item()) == 16  # 保证存在足够多真实可读句子与 mask 事件。
assert training_history[-1]["total_loss"] < training_history[0]["total_loss"] and all(row["generator_gradient_norm"] > 0.0 and row["discriminator_gradient_norm"] > 0.0 for row in training_history)  # 保证两套网络都执行真实 backward。
assert electra_balanced_accuracy > baseline_balanced_accuracy and replaced_recall > baseline_replaced_recall  # 保证同数据 balanced accuracy 和替换召回优于基线。
assert electra_balanced_accuracy >= 0.85 and generator_mlm_accuracy >= 0.80  # 保证两个训练目标都获得实质学习结果。
assert generator_parameters < discriminator_parameters and bool(final_rtd_labels[selected_mask].all())  # 保证小 Generator 配比与受控 hard-negative 生效。
assert int(wrong_edge_label[first_selected_position]) == 1 and int(correct_edge_label[first_selected_position]) == 0  # 保证 selected-versus-replaced 标签错误可复现并修正。